In [1]:
import pandas as pd
import numpy as np
import joblib
from rdkit import Chem
from rdkit.Chem import MACCSkeys, AllChem, Descriptors, Lipinski

In [2]:
# =====================================================================
# 1. HELPER FUNCTIONS (Fingerprint Extraction & Lipinski Filter)
# =====================================================================
def extract_fingerprints(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: 
            return None
        
        # 166-bit MACCS Keys
        maccs = np.array([int(b) for b in MACCSkeys.GenMACCSKeys(mol).ToBitString()])
        
        # 1024-bit Morgan Fingerprint (Radius = 3 for Morgan3 equivalent used in paper)
        morgan = np.array([int(b) for b in AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=1024).ToBitString()])
        
        return {"MACCS": maccs, "Morgan3": morgan}
    except:
        return None

def check_lipinski_criteria(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: 
            return False, 4
        
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Lipinski.NumHDonors(mol)
        h_acceptors = Lipinski.NumHAcceptors(mol)
        
        violations = 0
        if mw > 500: violations += 1
        if logp > 5: violations += 1
        if h_donors > 5: violations += 1
        if h_acceptors > 10: violations += 1
        
        # Paper allows a maximum of 1 violation
        return (violations <= 1), violations
    except:
        return False, 4

In [3]:
# =====================================================================
# 2. LOAD THE SAVED MODEL AND METADATA
# =====================================================================
print("--- LOADING PRE-TRAINED COMPONENT OBJECTS ---")

# Load metadata first to know which fingerprint and model format to look for
metadata = joblib.load("model_metadata.joblib")
winning_fp_name = metadata["winning_fingerprint_type"]
print(f"📖 Metadata loaded. Model expects fingerprint: {winning_fp_name}")

# Standardize the naming to find your saved model file
model_type_keyword = "rf" if "MACCS" in winning_fp_name and "RF" in winning_fp_name or "Morgan" in winning_fp_name else "svm"
# Alternatively, if you know the exact file name:
model_filename = "her2_champion_rf_model.joblib" # Change this if your saved file was named 'svm'

best_model = joblib.load(model_filename)
print(f"💾 Champion model successfully loaded from: {model_filename}")


# =====================================================================
# 3. LOAD THE PUBCHEM POOL AND EXTRACT RELEVANT FEATURES
# =====================================================================
print("\n--- PROCESSING PUBCHEM SCREENING DATA ---")
df_pubchem = pd.read_csv("data/pubchem_erbb2_screening_set.csv")
print(f"Loaded {len(df_pubchem)} molecules from PubChem file.")

# High-speed vectorized fingerprint calculation
print("Extracting chemical fingerprints...")
fps_extracted = df_pubchem['ConnectivitySMILES'].apply(extract_fingerprints)

screen_features = []
valid_indices = []

# Map metadata key to RDKit extraction dictionary keys

key_mapping = "MACCS" if "MACCS" in winning_fp_name else "Morgan3"

--- LOADING PRE-TRAINED COMPONENT OBJECTS ---
📖 Metadata loaded. Model expects fingerprint: Morgan3 (1024-bit)
💾 Champion model successfully loaded from: her2_champion_rf_model.joblib

--- PROCESSING PUBCHEM SCREENING DATA ---
Loaded 4593 molecules from PubChem file.
Extracting chemical fingerprints...


[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerator
[09:33:01] DEPRECATION WARNING: please use MorganGenerat

In [4]:
for idx, fp in enumerate(fps_extracted):
    if fp is not None:
        screen_features.append(fp[key_mapping])
        valid_indices.append(idx)

# Align screening dataframe to match only molecules that successfully parsed
df_screening = df_pubchem.iloc[valid_indices].copy()
X_screen = np.array(screen_features)


# =====================================================================
# 4. RUN INFERENCE & FILTERING (VIRTUAL SCREENING SELECTION)
# =====================================================================
print("\n--- RUNNING INFERENCE (FILTER 1: AI CLASSIFICATION) ---")
# The loaded machine learning model makes predictions directly on the features
df_screening['ML_Prediction'] = best_model.predict(X_screen)

# Keep only the rows flagged as Active (1)
df_actives = df_screening[df_screening['ML_Prediction'] == 1].copy()
print(f"-> AI Filter complete: {len(df_actives)} structures predicted active.")

print("\n--- DRUG-LIKENESS SCREENING (FILTER 2: LIPINSKI'S RULE) ---")
pass_lipinski_list = []
violation_counts = []

for idx, row in df_actives.iterrows():
    is_drug_like, total_viol = check_lipinski_criteria(row['ConnectivitySMILES'])
    pass_lipinski_list.append(is_drug_like)
    violation_counts.append(total_viol)

df_actives['Passes_Lipinski'] = pass_lipinski_list
df_actives['Lipinski_Violations'] = violation_counts

# Filter to keep compounds with 0 or 1 violation max
df_final_leads = df_actives[df_actives['Passes_Lipinski'] == True].copy()


# =====================================================================
# 5. EXPORT FINAL DISCOVERED LEADS
# =====================================================================
output_csv = "data/final_drug_like_her2_hits.csv"
df_final_leads.to_csv(output_csv, index=False)

print(f"\n🎉 Deployment Process Complete!")
print(f"Selected {len(df_final_leads)} potential HER2 active drug-like hits.")
print(f"Results exported successfully to '{output_csv}' for molecular docking analysis.")


--- RUNNING INFERENCE (FILTER 1: AI CLASSIFICATION) ---
-> AI Filter complete: 3524 structures predicted active.

--- DRUG-LIKENESS SCREENING (FILTER 2: LIPINSKI'S RULE) ---

🎉 Deployment Process Complete!
Selected 2330 potential HER2 active drug-like hits.
Results exported successfully to 'data/final_drug_like_her2_hits.csv' for molecular docking analysis.


In [9]:
print(df_final_leads)

            CID                                 ConnectivitySMILES  \
0       5329249  CC1=C(NC(=C1)C(=O)N2CCOCC2)C=C3C4=C(NC3=O)N=CN...   
1      10300451  CC1=C(NC(=C1)C(=O)NCCN2CCOCC2)C=C3C4=C(NC3=O)N...   
2       5329232  CC1=CC(=C(N1)C=C2C3=C(NC2=O)N=CN=C3NC4=CC(=C(C...   
3       5329244  CCN(CC)CCNC(=O)C1=CC(=C(N1)C=C2C3=C(NC2=O)N=CN...   
4       5329243  CCN1CCN(CC1)C(=O)C2=CC(=C(N2)C=C3C4=C(NC3=O)N=...   
...         ...                                                ...   
4586  170966901  CC1=C(C=CC(=C1)NC2=NC=NN3C2=C(C=C3)C4CCCN(CC4)...   
4587  170966937  CC1=C(C=CC(=C1)NC2=NC=NN3C2=C(C=C3)C4CC5(C4)CN...   
4589  170967053  CC1=C(C=CC(=C1)NC2=NC=NN3C2=C(C=C3)N4CCN(CC4)C...   
4590  178185076  CC1=C(C=CC(=C1)NC2=NC=NN3C2=C(C=C3)C4=CN(N=C4)...   
4591  160283094  CC1=C(C=CC(=C1)NC2=NC=NC3=CN=C(N=C32)N4CCC(CC4...   

      ML_Prediction  Passes_Lipinski  Lipinski_Violations  
0                 1             True                    0  
1                 1             True   

In [10]:
# =====================================================================
# 4. RUN INFERENCE (FILTER 1: AI PROBABILITY RANKING)
# =====================================================================
print("\n--- RUNNING INFERENCE (RANKING BY ACTIVE PROBABILITY) ---")

# CRITICAL CHANGE: Instead of predict(), we use predict_proba() to get confidence percentages
# predict_proba outputs [probability_of_0, probability_of_1]. We extract index 1 (Active).
probabilities = best_model.predict_proba(X_screen)[:, 1]

# Attach probabilities to our processed dataset
df_screening['Active_Probability'] = probabilities

# Sort all molecules from highest probability (most likely to be active) to lowest
df_ranked = df_screening.sort_values(by='Active_Probability', ascending=False).copy()
print(f"Successfully ranked {len(df_ranked)} compounds by model confidence.")


# =====================================================================
# 5. LIPINSKI DRUG-LIKENESS FILTERING & TOP 20 EXTRACTION
# =====================================================================
print("\n--- EXTRACTING THE TOP 20 DRUG-LIKE CANDIDATES ---")

top_20_leads = []

# Loop through the ranked molecules one by one until we find 20 that pass Lipinski's rules
for idx, row in df_ranked.iterrows():
    # If we have already successfully collected 20 drug-like compounds, stop the loop!
    if len(top_20_leads) == 20:
        break
        
    is_drug_like, total_viol = check_lipinski_criteria(row['ConnectivitySMILES'])
    
    if is_drug_like:
        # Create a dictionary containing all the data we want to save
        lead_data = {
            "CID": row['CID'],
            "ConnectivitySMILES": row['ConnectivitySMILES'],
            "AI_Confidence_Score": row['Active_Probability'],
            "Lipinski_Violations": total_viol
        }
        top_20_leads.append(lead_data)

# Convert our list of top 20 candidates into a final clean DataFrame
df_top_20 = pd.DataFrame(top_20_leads)


# =====================================================================
# 6. EXPORT AND PREPARE FOR MOLECULAR DOCKING
# =====================================================================
output_csv = "data/top_20_her2_docking_candidates.csv"
df_top_20.to_csv(output_csv, index=False)

print("\n================== TOP 20 CANDIDATES FOR DOCKING ==================")
print(df_top_20.to_string(index=False, formatters={'AI_Confidence_Score': '{:.4f}'.format}))
print("===================================================================\n")

print(f"🎉 Process Complete! Saved exactly {len(df_top_20)} top hits to '{output_csv}'.")
print("You can now download the 3D structures (.SDF/.PDB) for these 20 CIDs from PubChem to begin your molecular docking!")


--- RUNNING INFERENCE (RANKING BY ACTIVE PROBABILITY) ---
Successfully ranked 4593 compounds by model confidence.

--- EXTRACTING THE TOP 20 DRUG-LIKE CANDIDATES ---

================== TOP 20 CANDIDATES FOR DOCKING ==================
      CID                                                             ConnectivitySMILES AI_Confidence_Score  Lipinski_Violations
130277012     CN(C)C(=O)CC1=CC(=C(C=C1)NC(=O)C2=NN(C3=NC=NC(=C23)N)C4CCCN(C4)C(=O)C=C)Cl              0.9806                    1
130276342    CC1=C(C=CC(=C1Cl)NC(=O)C2=NN(C3=NC=NC(=C23)N)C4CCCN(C4)C(=O)C=C)CC(=O)N(C)C              0.9769                    1
130277015     CN(C)C(=O)CC1=CC(=C(C=C1)NC(=O)C2=NN(C3=NC=NC(=C23)N)C4CCCN(C4)C(=O)C=C)OC              0.9758                    1
130276915  CN(C)C(=O)CC1=C(C(=C(C=C1)NC(=O)C2=NN(C3=NC=NC(=C23)N)C4CCCN(C4)C(=O)C=C)Cl)F              0.9756                    1
130275885  CN(C)C1=C(C=CC(=C1)CC(=O)N(C)C)NC(=O)C2=NN(C3=NC=NC(=C23)N)C4CCCN(C4)C(=O)C=C              0.9746      

In [16]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
# from meeko import MoleculePreparation, PDBQTWriter

from meeko import MoleculePreparation
# Import the updated legacy writer name
from meeko import PDBQTWriterLegacy 
from rdkit import Chem


# Create an output directory for your prepared 3D ligands
output_dir = "data/prepared_ligands"
os.makedirs(output_dir, exist_ok=True)

# Load your top 20 candidate file
df_top_20 = pd.read_csv("data/top_20_her2_docking_candidates.csv")

print(f"--- PREPARING 3D GEOMETRIES FOR {len(df_top_20)} LIGANDS ---")

--- PREPARING 3D GEOMETRIES FOR 20 LIGANDS ---


In [17]:
df_top_20

,CID,ConnectivitySMILES,AI_Confidence_Score,Lipinski_Violations
0,130277012,CN(C)C(=O)CC1=CC(=C(C=C1)NC(=O)C2=NN(C3=NC=NC(...,0.980625,1
1,130276342,CC1=C(C=CC(=C1Cl)NC(=O)C2=NN(C3=NC=NC(=C23)N)C...,0.976875,1
2,130277015,CN(C)C(=O)CC1=CC(=C(C=C1)NC(=O)C2=NN(C3=NC=NC(...,0.975833,1
3,130276915,CN(C)C(=O)CC1=C(C(=C(C=C1)NC(=O)C2=NN(C3=NC=NC...,0.975625,1
4,130275885,CN(C)C1=C(C=CC(=C1)CC(=O)N(C)C)NC(=O)C2=NN(C3=...,0.974625,1
5,155755925,CN(C)CC=CC(=O)N1CCOC2=C1C=CC3=C2C(=NC=N3)NC4=C...,0.973393,1
6,130275835,CN(C)C(=O)CC1=C(C(=C(C=C1)NC(=O)C2=NN(C3=NC=NC...,0.972847,1
7,130276918,CN(C)C(=O)CC1=CC(=C(C=C1)NC(=O)C2=NN(C3=NC=NC(...,0.972000,1
8,130276350,CC1=C(C=CC(=C1)OC(=O)N2CCCC2)NC(=O)C3=NN(C4=NC...,0.971667,1
9,155755958,C=CC(=O)N1CCOC2=C1C=CC3=C2C(=NC=N3)NC4=CC=C(C=...,0.970833,0


In [20]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation
from meeko import PDBQTWriterLegacy 

# Create an output directory for your prepared 3D ligands
output_dir = "data/prepared_ligands"
os.makedirs(output_dir, exist_ok=True)

# Load your top 20 candidate file
df_top_20 = pd.read_csv("data/top_20_her2_docking_candidates.csv")

print(f"--- PREPARING 3D GEOMETRIES FOR {len(df_top_20)} LIGANDS ---")

for idx, row in df_top_20.iterrows():
    cid = row['CID']
    smiles = row['ConnectivitySMILES']
    
    print(f"Processing CID: {cid}...")
    
    # 1. Convert SMILES string to RDKit Molecule object & add Explicit Hydrogens
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"❌ Failed to parse SMILES for CID {cid}")
        continue
    mol_with_h = Chem.AddHs(mol)
    
    # 2. Generate Initial 3D Coordinates using Distance Geometry (ETKDGv3 method)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    embed_status = AllChem.EmbedMolecule(mol_with_h, params)
    
    if embed_status == -1:
        print(f"⚠️ Initial 3D embedding failed for CID {cid}. Trying fallback options...")
        AllChem.EmbedMolecule(mol_with_h, randomSeed=42, useRandomCoords=True)
    
    # 3. Optimize Geometry using the Universal Force Field (UFF)
    # This relaxes bond lengths and structural angles into realistic 3D states
    optimization_status = AllChem.UFFOptimizeMolecule(mol_with_h, maxIters=500)
    if optimization_status != 0:
        print(f"⚠️ Energy minimization didn't fully converge for CID {cid}, using best layout.")
        
    # 4. Convert RDKit object to AutoDock PDBQT format using Meeko
    # Meeko handles flexible rotatable bonds and atom type definitions automatically
    preparer = MoleculePreparation()
    
    # FIX 1: .prepare() generates and returns a list of setup objects
    mol_setups = preparer.prepare(mol_with_h)
    
    if not mol_setups:
        print(f"❌ Meeko setup generation returned empty for CID {cid}")
        continue
        
    # For standard single-ligand pipelines, grab the first configuration
    setup = mol_setups[0]
    
    # FIX 2: Pass the specific setup object (instead of the factory preparer tool)
    writer_output = PDBQTWriterLegacy.write_string(setup)
    
    # FIX 3: Safe unpacking handling across varying Meeko package updates
    if isinstance(writer_output, tuple):
        pdbqt_string = writer_output[0]
    else:
        pdbqt_string = writer_output
    
    if pdbqt_string:
        output_path = os.path.join(output_dir, f"ligand_{cid}.pdbqt")
        with open(output_path, "w") as f:
            f.write(pdbqt_string)
    else:
        print(f"❌ Meeko PDBQT string output generation empty for CID {cid}")

print(f"\n🎉 Finished! All successfully prepared ligand .pdbqt files are saved in '{output_dir}/'")


--- PREPARING 3D GEOMETRIES FOR 20 LIGANDS ---
Processing CID: 130277012...
Processing CID: 130276342...
Processing CID: 130277015...
Processing CID: 130276915...
Processing CID: 130275885...
Processing CID: 155755925...
Processing CID: 130275835...
Processing CID: 130276918...
Processing CID: 130276350...
Processing CID: 155755958...
Processing CID: 130277020...
Processing CID: 130275856...
Processing CID: 130276911...
Processing CID: 130276411...
Processing CID: 130275249...
Processing CID: 155756034...
Processing CID: 155755966...
Processing CID: 130276952...
Processing CID: 130276381...
Processing CID: 155755998...

🎉 Finished! All successfully prepared ligand .pdbqt files are saved in 'data/prepared_ligands/'
